# Entrenamiento PKLot: clasificador libre/ocupado con MobileNetV2
Notebook pensado para correr en Google Colab. Entrena un clasificador binario
sobre patches del dataset PKLot y exporta el modelo final a Google Drive.

In [ ]:
# Celda 1: Instalacion/import de dependencias
!pip install -q tensorflow scikit-learn matplotlib seaborn

import os
import json
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

In [ ]:
# Celda 2: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Celda 3: Configuracion (editar estas variables antes de correr el notebook)
# Ruta a la carpeta del dataset PKLot dentro de tu Drive.
DATASET_PATH = "/content/drive/MyDrive/PKLot"
# Carpeta de Drive donde se va a guardar el modelo entrenado.
OUTPUT_PATH = "/content/drive/MyDrive/PKLot_output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
RANDOM_SEED = 42

## Exploracion de la estructura de carpetas
PKLot puede venir con estructuras distintas segun como se descargue
(por ejemplo `PKLot/PKLotSegmented/UFPR04/Cloudy/2012-09-12/Empty/...`).
Esta celda recorre `DATASET_PATH` y muestra el arbol de carpetas y cuantas
imagenes Empty/Occupied encuentra, para verificar el path ANTES de entrenar.

In [ ]:
# Celda 4: Explorar la estructura real del dataset
def print_tree(root, max_depth=4, _depth=0):
    """Imprime el arbol de carpetas hasta max_depth niveles de profundidad."""
    if _depth > max_depth:
        return
    root = Path(root)
    entries = sorted([p for p in root.iterdir() if p.is_dir()])
    for entry in entries:
        print("  " * _depth + f"- {entry.name}/")
        print_tree(entry, max_depth=max_depth, _depth=_depth + 1)

print(f"Explorando: {DATASET_PATH}\n")
print_tree(DATASET_PATH)

empty_count = sum(1 for _ in Path(DATASET_PATH).rglob("Empty/*") if _.is_file())
occupied_count = sum(1 for _ in Path(DATASET_PATH).rglob("Occupied/*") if _.is_file())
empty_count += sum(1 for _ in Path(DATASET_PATH).rglob("empty/*") if _.is_file())
occupied_count += sum(1 for _ in Path(DATASET_PATH).rglob("occupied/*") if _.is_file())
print(f"\nImagenes encontradas -> Empty: {empty_count} | Occupied: {occupied_count}")
assert empty_count > 0 and occupied_count > 0, "No se encontraron imagenes Empty/Occupied en DATASET_PATH. Revisa la ruta."

## Carga del dataset y split train/val/test
Buscamos todas las imagenes bajo carpetas `Empty`/`Occupied` (sin importar
mayusculas) y armamos pares (ruta, label). Si detectamos subcarpetas con
formato de fecha (PKLot trae `Cloudy/2012-09-12/...`), agrupamos el split por
fecha para que todos los frames de un mismo dia queden en el mismo split y
evitar data leakage entre frames del mismo video. Si no hay esa estructura,
hacemos split aleatorio estratificado por clase.

In [ ]:
# Celda 5: Recolectar imagenes y labels
def collect_image_label_pairs(dataset_path):
    """Recorre dataset_path y devuelve lista de (ruta, label, date_key).

    date_key es la carpeta inmediatamente superior a Empty/Occupied si su
    nombre tiene forma de fecha (YYYY-MM-DD), o None si no se pudo inferir.
    """
    import re

    date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
    pairs = []
    for label_name, label in [("Empty", 0), ("empty", 0), ("Occupied", 1), ("occupied", 1)]:
        for image_path in Path(dataset_path).rglob(f"{label_name}/*"):
            if not image_path.is_file():
                continue
            parent_date_dir = image_path.parent.parent.name
            date_key = parent_date_dir if date_pattern.match(parent_date_dir) else None
            pairs.append((str(image_path), label, date_key))
    return pairs


pairs = collect_image_label_pairs(DATASET_PATH)
print(f"Total de imagenes encontradas: {len(pairs)}")

has_date_structure = all(date_key is not None for _, _, date_key in pairs) and len(pairs) > 0
print(f"Estructura por fecha detectada: {has_date_structure}")

In [ ]:
# Celda 6: Split train/val/test (70/15/15)
random.seed(RANDOM_SEED)

if has_date_structure:
    # Split por fecha: todos los frames de un mismo dia van al mismo split.
    dates = sorted(set(date_key for _, _, date_key in pairs))
    random.shuffle(dates)
    n = len(dates)
    train_dates = set(dates[: int(0.7 * n)])
    val_dates = set(dates[int(0.7 * n): int(0.85 * n)])
    test_dates = set(dates[int(0.85 * n):])

    train_pairs = [(p, l) for p, l, d in pairs if d in train_dates]
    val_pairs = [(p, l) for p, l, d in pairs if d in val_dates]
    test_pairs = [(p, l) for p, l, d in pairs if d in test_dates]
else:
    # Fallback: split aleatorio estratificado por clase.
    paths = [p for p, l, _ in pairs]
    labels = [l for _, l, _ in pairs]
    train_paths, rest_paths, train_labels, rest_labels = train_test_split(
        paths, labels, test_size=0.3, stratify=labels, random_state=RANDOM_SEED
    )
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        rest_paths, rest_labels, test_size=0.5, stratify=rest_labels, random_state=RANDOM_SEED
    )
    train_pairs = list(zip(train_paths, train_labels))
    val_pairs = list(zip(val_paths, val_labels))
    test_pairs = list(zip(test_paths, test_labels))


def count_by_label(split_pairs):
    counts = defaultdict(int)
    for _, label in split_pairs:
        counts[label] += 1
    return dict(counts)


print(f"Train: {len(train_pairs)} imagenes -> {count_by_label(train_pairs)}")
print(f"Val:   {len(val_pairs)} imagenes -> {count_by_label(val_pairs)}")
print(f"Test:  {len(test_pairs)} imagenes -> {count_by_label(test_pairs)}")

## Pipeline de datos con augmentation
Armamos `tf.data.Dataset` para cada split. El de train tiene augmentation
(flip horizontal, brillo, rotacion leve) ya que PKLot tiene variacion de
clima/luz; val y test no llevan augmentation.

In [ ]:
# Celda 7: Construccion de los tf.data.Dataset
def load_and_resize(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = image / 255.0
    return image, tf.cast(label, tf.float32)


def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.keras.layers.RandomRotation(0.04)(tf.expand_dims(image, 0))[0]  # ~ +/-15 grados
    return image, label


def make_dataset(split_pairs, training):
    paths = [p for p, _ in split_pairs]
    labels = [l for _, l in split_pairs]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_resize, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=len(split_pairs), seed=RANDOM_SEED)
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_pairs, training=True)
val_ds = make_dataset(val_pairs, training=False)
test_ds = make_dataset(test_pairs, training=False)

## Modelo: transfer learning con MobileNetV2
Usamos MobileNetV2 preentrenado en ImageNet como extractor de caracteristicas
(congelado), con una cabeza de clasificacion binaria simple encima.

In [ ]:
# Celda 8: Definicion del modelo
base_model = MobileNetV2(input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
)

model.summary()

## Entrenamiento con callbacks
EarlyStopping evita sobreentrenar, ModelCheckpoint guarda el mejor modelo
visto durante el entrenamiento, y ReduceLROnPlateau baja el learning rate
cuando el val_loss se estanca.

In [ ]:
# Celda 9: Entrenamiento
checkpoint_path = os.path.join(OUTPUT_PATH, "best_checkpoint.h5")

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## Evaluacion sobre el set de test
Calculamos accuracy, precision, recall, F1 y la matriz de confusion sobre
imagenes que el modelo nunca vio durante el entrenamiento.

In [ ]:
# Celda 10: Evaluacion en test
y_true = []
y_pred = []

for images, labels in test_ds:
    probabilities = model.predict(images, verbose=0).flatten()
    y_pred.extend((probabilities >= 0.5).astype(int).tolist())
    y_true.extend(labels.numpy().astype(int).tolist())

print(classification_report(y_true, y_pred, target_names=["Empty", "Occupied"]))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Empty", "Occupied"]).plot(cmap="Blues")
plt.title("Matriz de confusion - Test set")
plt.show()

## Curvas de entrenamiento
Graficamos loss y accuracy de train vs val por epoch para ver si hubo
overfitting o underfitting.

In [ ]:
# Celda 11: Graficos de loss y accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## Exportar el modelo final a Google Drive
Guardamos el modelo entrenado en `OUTPUT_PATH` en formato `.h5`, listo para
descargar y usar localmente con `inference/detect.py`.

In [ ]:
# Celda 12: Exportar modelo final
final_model_path = os.path.join(OUTPUT_PATH, "modelo_pklot.h5")
model.save(final_model_path)
print(f"Modelo guardado en: {final_model_path}")
print("Descargalo desde Google Drive y usalo con --model en inference/detect.py")